# External dbPTM/PTMGPT2 validation v2

This notebook constructs a leakage-audited external cohort before it performs any model inference. Version 2 corrects the pre-inference MMseqs2 identity audit by requesting alignment backtraces and fails closed if reported alignments contain only zero identity counts. It then evaluates the already frozen exact-MMUbiPred/context equal-probability ensemble once. Do not change the fusion weights, threshold, exclusion rules, or checkpoints after seeing external predictions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Store a fine-grained read-only GitHub token in Colab Secrets as `GITHUB_TOKEN`. An `HF_TOKEN` is recommended for loading ESM-2.

In [ ]:
from pathlib import Path
import os, subprocess, sys
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN, HF_TOKEN = secret('GITHUB_TOKEN'), secret('HF_TOKEN')
git_env = os.environ.copy()
git_env['GIT_TERMINAL_PROMPT'] = '0'
askpass = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; *) echo "$GITHUB_TOKEN";; esac\n')
    askpass.chmod(0o700)
    git_env.update({'GIT_ASKPASS': str(askpass), 'GITHUB_TOKEN': GITHUB_TOKEN})

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run(['git', 'clone', '--recurse-submodules', 'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git', str(REPO_ROOT)], check=True, env=git_env)
    else:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True, env=git_env)
        subprocess.run(['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'], check=True, env=git_env)
finally:
    askpass.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = '/content/drive/MyDrive/MMUbiPred-research/huggingface-cache'
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip install -q "tf-keras==2.19.0"
%pip uninstall -y torchao
!apt-get update -qq && apt-get install -y -qq mmseqs2

In [ ]:
import json, torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before external inference.')
print('GPU:', torch.cuda.get_device_name(0))

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

## Verify the frozen source artifacts

The training UniProt cache and seven-epoch context run are reused from notebook 09. The released H5 model may remain in the replication artifact directory.

In [ ]:
CONFIG = Path('experiment/configs/external_dbptm_ptmgpt2_v2.json')
EXTERNAL_ROOT = Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-ptmgpt2-v2')
COHORT_DIR = EXTERNAL_ROOT / 'cohort'
EXTERNAL_SEQUENCE_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/external/dbptm-ptmgpt2-v1/uniprot_external_sequences.json')
TRAINING_SEQUENCE_CACHE = Path('/content/drive/MyDrive/MMUbiPred-research/context/uniprot_training_sequences.json')
CONTEXT_RUN_DIR = Path('/content/drive/MyDrive/MMUbiPred-research/experiments/mmubipred-context-only-final-v1-seed42')
PAPER_MODEL = Path('/content/drive/MyDrive/MMUbiPred-replication/artifacts/DeepUBI_AAindex_One_Hot_Emb_drop_out2423.h5')
EVALUATION_DIR = EXTERNAL_ROOT / 'evaluation'

for required in [CONFIG, TRAINING_SEQUENCE_CACHE, CONTEXT_RUN_DIR / 'best.pt', CONTEXT_RUN_DIR / 'refit_manifest.json', CONTEXT_RUN_DIR / 'refit_summary.json', PAPER_MODEL]:
    if not required.exists():
        raise FileNotFoundError(required)
EXTERNAL_ROOT.mkdir(parents=True, exist_ok=True)
print('All frozen source artifacts were found.')

## Preflight tests

In [ ]:
run_live([sys.executable, '-u', '-m', 'unittest', 'discover', '-s', 'experiment/tests'])

## Retrieve external full-protein sequences

This is resumable. It retrieves only missing accessions and stores UniProt release metadata on Drive. It does not run either model.

In [ ]:
run_live([
    sys.executable, '-u', 'experiment/scripts/fetch_external_benchmark_sequences.py',
    '--benchmark', 'replication/MMUbiPred/benchmark.csv',
    '--output', EXTERNAL_SEQUENCE_CACHE,
])

## Construct and freeze the cohort

This validates published site coordinates, removes accession/site/window overlap, performs the frozen MMseqs2 homology filter, and writes `cohort_lock.json`. It refuses to overwrite an existing lock and performs no inference.

In [ ]:
LOCK_PATH = COHORT_DIR / 'cohort_lock.json'
FEASIBILITY_PATH = COHORT_DIR / 'cohort_feasibility.json'
if LOCK_PATH.exists():
    print('Existing frozen cohort found; construction was not repeated.')
else:
    try:
        run_live([
            sys.executable, '-u', 'experiment/scripts/prepare_external_benchmark.py',
            '--config', CONFIG,
            '--benchmark', 'replication/MMUbiPred/benchmark.csv',
            '--external-sequence-cache', EXTERNAL_SEQUENCE_CACHE,
            '--training-sequence-cache', TRAINING_SEQUENCE_CACHE,
            '--released-model', PAPER_MODEL,
            '--context-run-dir', CONTEXT_RUN_DIR,
            '--output-dir', COHORT_DIR,
        ])
    except subprocess.CalledProcessError:
        if FEASIBILITY_PATH.exists():
            print('Cohort was not frozen. Pre-inference feasibility audit:')
            display(json.loads(FEASIBILITY_PATH.read_text()))
        raise

cohort_lock = json.loads(LOCK_PATH.read_text())
display({
    'status': cohort_lock['status'],
    'validation': cohort_lock['validation'],
    'exact_leakage_filter': cohort_lock['exact_leakage_filter'],
    'homology_filter': cohort_lock['homology_filter'],
    'final_cohort': cohort_lock['final_cohort'],
})

## Freeze point

Stop here initially and send the displayed cohort audit for review. No external predictions exist yet. Once the audit passes, set the acknowledgement below to `True`. Evaluation is one-time and its metrics file prevents accidental repetition.

In [ ]:
RUN_ONE_TIME_EXTERNAL_INFERENCE = False  # Change to True only after reviewing the frozen cohort audit.

RESULTS_PATH = EVALUATION_DIR / 'external_test_metrics.json'
if RESULTS_PATH.exists():
    print('Completed external result already exists; inference was not repeated.')
elif not RUN_ONE_TIME_EXTERNAL_INFERENCE:
    print('External inference remains sealed. Review cohort_lock.json first.')
else:
    run_live([
        sys.executable, '-u', 'experiment/scripts/evaluate_external_benchmark.py',
        '--config', CONFIG,
        '--cohort-dir', COHORT_DIR,
        '--model-file', PAPER_MODEL,
        '--context-run-dir', CONTEXT_RUN_DIR,
        '--output-dir', EVALUATION_DIR,
        '--batch-size', '32',
        '--num-workers', '0',
        '--allow-external-test',
    ])

## Inspect the completed external result

In [ ]:
if not RESULTS_PATH.exists():
    print('No external metrics exist. The evaluation is still sealed.')
else:
    result = json.loads(RESULTS_PATH.read_text())
    display({
        'exact_released_mmubipred': result['exact_released_mmubipred'],
        'frozen_context_model': result['frozen_context_model'],
        'equal_probability_ensemble': result['equal_probability_ensemble'],
        'ensemble_minus_exact_mmubipred': result['ensemble_minus_exact_mmubipred'],
        'mcnemar_accuracy': result['paired_ensemble_vs_exact_mmubipred']['mcnemar_accuracy'],
        'bootstrap_metric_differences': result['paired_ensemble_vs_exact_mmubipred']['bootstrap']['metrics'],
    })